In [1]:
from rbfPUCurlNoise.triang_mesh     import TriMesh, refine_mesh_1to4
from rbfPUCurlNoise.patches       import PatchCoverage
from rbfPUCurlNoise.rbf           import RBF
from perlin_noise import PerlinNoise
from rbfPUCurlNoise.potential import NoisePotential2D
import triangle
import numpy as np


In [2]:
noise_seed = 7
noise = PerlinNoise(octaves=3, seed=noise_seed)

In [3]:
def analytic_noise_function(noise, x, y, freq):
    noise_val = noise([x * freq, y * freq])
    return noise_val

## alienbs

In [5]:
alienbs = triangle.load("..\\meshes\\triangle_files", "alienbs")
coarse_alienbs_mesh = triangle.triangulate(alienbs, "pq30a150")

print(coarse_alienbs_mesh["vertices"].min(), coarse_alienbs_mesh["vertices"].max())
mins = coarse_alienbs_mesh["vertices"].min(axis=0)
maxs = coarse_alienbs_mesh["vertices"].max(axis=0)


# Normalizing mesh vertices range
coarse_alienbs_mesh["vertices"] = 2 * (coarse_alienbs_mesh["vertices"] - mins) / (maxs - mins) - 1
print(coarse_alienbs_mesh["vertices"].min(), coarse_alienbs_mesh["vertices"].max())

coarse_alienbs_mesh_obj = TriMesh(coarse_alienbs_mesh)

#plot_mesh(alienbs_mesh)

-1001.81 828.5
-1.0 1.0


In [6]:
freq = 2
mesh_noise = coarse_alienbs_mesh_obj.generate_mesh_noise_scalar_field(noise, freq)

In [7]:

coverage = PatchCoverage(coarse_alienbs_mesh_obj, H=0.2, delta=0.2, min_nodes_per_patch=4)
coverage.summary() # Resumo da cobertura

potential_alien = NoisePotential2D(coarse_alienbs_mesh_obj, coverage, RBF(epsilon=1.0))
potential_alien.fit(mesh_noise)

166
168
61
31
179
242
57
105
169
156
140
192
157
74
86
172
173
178
178
180
181
198
167
200
38
104
154
152
134
139
168
192
136
100
7
55
111
175
129
105
145
121
27
0
118
175
139
143
185
200
123
128
129
66
144
186
105
18
148
182
28
57
149
159
50
129
145
117
176
181
141
182
146
92
87
67
51
97
115
90
102
111
85
180
100
8
0
0
72
142
109

Coverage Summary:
  n_patches           : 81
  H                   : 0.2
  delta               : 0.2
  min_nodes           : 50
  max_nodes           : 242
  mean_nodes          : 136.17283950617283
  min_coverage        : 1
  max_coverage        : 4
  mean_coverage       : 2.068254265891618
  full_coverage       : True


In [8]:
ver_ref, tri_ref = refine_mesh_1to4(coarse_alienbs_mesh["vertices"], coarse_alienbs_mesh["triangles"])
refined_alien_mesh = {}
refined_alien_mesh["vertices"] = ver_ref
refined_alien_mesh["triangles"] = tri_ref

refined_alien_mesh_obj = TriMesh(refined_alien_mesh)
#plot_mesh(mesh_refined)

In [9]:
fine_scalar_field = potential_alien.evaluate(refined_alien_mesh["vertices"])
#plot_noise_mesh(alienbs_mesh, fine_scalar_field)

In [ ]:
# Interpolation error
exact_fine_scalar_field = []

for (x, y) in refined_alien_mesh["vertices"]:
    analytic_val = analytic_noise_function(noise, x, y, freq)
    exact_fine_scalar_field.append(analytic_val)

exact_fine_scalar_field = np.asarray(exact_fine_scalar_field)
err   = np.abs(exact_fine_scalar_field - fine_scalar_field)
max = float(err.max())
l2 = float(np.sqrt(np.mean(err**2)))
print('Max:', max, 'L2:', l2)

Max: 0.005806033901581298 L2: 0.00046537085093383804


In [ ]:
refined_alien_mesh_obj.save_mesh_paraview(refined_alien_mesh, phi=fine_scalar_field, out_name="alienbs_interpolation.vtu")

0 20404 20405


## starfish

In [ ]:
from scipy.io      import loadmat

freq = 1
data = loadmat('starfish.mat')
pts = data['X']
print(pts.min(), pts.max())
tri = data['tri'] - 1

coarse_starfish_mesh = {}
coarse_starfish_mesh["vertices"] = pts
coarse_starfish_mesh["triangles"] = tri

-1.0 1.0


In [ ]:
coarse_starfish_mesh_obj = TriMesh(coarse_starfish_mesh)

freq = 1
mesh_noise = coarse_starfish_mesh_obj.generate_mesh_noise_scalar_field(noise, freq)

In [ ]:

coverage = PatchCoverage(coarse_starfish_mesh_obj, H=0.2, delta=0.2, min_nodes_per_patch=4)
coverage.summary() # Resumo da cobertura

potential_starfish = NoisePotential2D(coarse_starfish_mesh_obj, coverage, RBF(epsilon=1.0))
potential_starfish.fit(mesh_noise)



Resumo da cobertura:
  n_patches           : 75
  H                   : 0.2
  delta               : 0.2
  min_nodes           : 13
  max_nodes           : 58
  mean_nodes          : 38.93333333333333
  min_coverage        : 1
  max_coverage        : 4
  mean_coverage       : 2.1128798842257597
  full_coverage       : True


In [ ]:
ver_ref, tri_ref = refine_mesh_1to4(coarse_starfish_mesh["vertices"], coarse_starfish_mesh["triangles"])
refined_starfish_mesh = {}
refined_starfish_mesh["vertices"] = ver_ref
refined_starfish_mesh["triangles"] = tri_ref

refined_starfish_mesh_obj = TriMesh(refined_starfish_mesh)

In [ ]:
fine_scalar_field = potential_starfish.evaluate(refined_starfish_mesh["vertices"])

In [ ]:
# Interpolation error
exact_fine_scalar_field = []

for (x, y) in refined_starfish_mesh["vertices"]:
    analytic_val = analytic_noise_function(noise, x, y, freq)
    exact_fine_scalar_field.append(analytic_val)

exact_fine_scalar_field = np.asarray(exact_fine_scalar_field)
err   = np.abs(exact_fine_scalar_field - fine_scalar_field)
max = float(err.max())
l2 = float(np.sqrt(np.mean(err**2)))
print('Max:', max, 'L2:', l2)

Max: 0.010099631772564496 L2: 0.000591083250868254


In [ ]:
refined_starfish_mesh_obj.save_mesh_paraview(refined_starfish_mesh, phi=fine_scalar_field, out_name="starfish_interpolation.vtu")

0 5396 5397
